In [0]:
from pyspark.sql.functions import col

### **Inserting Bronze Table into Silver View**

In [0]:
spark.sql("""
          select *,
                  current_timestamp() as processDate
          from retailsales.bronze.bronze_table
          """).createOrReplaceTempView("silver_source")

In [0]:
%sql
select * from silver_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,Electronics,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:01:56.019Z
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,Electronics,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:01:56.019Z
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,Footwear,1,129.99,Credit Card,Canada,2026-03-04,2026-03-04T16:01:56.019Z
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,Electronics,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:01:56.019Z
1005,2026-03-01,5,Emma Wilson,null,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:01:56.019Z
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,Electronics,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:01:56.019Z


In [0]:
df = spark.sql("SELECT * FROM silver_source")

In [0]:
display(df)

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,Electronics,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:01:57.199Z
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,Electronics,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:01:57.199Z
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,Footwear,1,129.99,Credit Card,Canada,2026-03-04,2026-03-04T16:01:57.199Z
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,Electronics,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:01:57.199Z
1005,2026-03-01,5,Emma Wilson,null,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:01:57.199Z
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,Electronics,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:01:57.199Z


### **Removing Duplicates**

In [0]:
df = df.dropDuplicates(subset=['order_id'])

In [0]:
print("Before:", spark.table("retailsales.bronze.bronze_table").count())
print("After:", df.count())

Before: 6
After: 6


### **Handling NULLs**

In [0]:
from pyspark.sql.functions import col


In [0]:
df.filter(col("customer_email").isNull()).display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate
1005,2026-03-01,5,Emma Wilson,null,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:01:58.812Z


In [0]:
df = df.fillna({"customer_email": "unknown"})

In [0]:
df.filter(col("payment_type").isNull()).display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate


In [0]:
df = df.fillna({"payment_type": "Cash"})

In [0]:
df.display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,Electronics,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:00.273Z
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,Electronics,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:02:00.273Z
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,Footwear,1,129.99,Credit Card,Canada,2026-03-04,2026-03-04T16:02:00.273Z
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,Electronics,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:02:00.273Z
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:02:00.273Z
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,Electronics,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:00.273Z


In [0]:
df = df.fillna({"unit_price": 100})

### **Dropping rows which has null values**

In [0]:
df = df.dropna('any')

In [0]:
df.display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,Electronics,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:01.197Z
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,Electronics,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:02:01.197Z
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,Footwear,1,129.99,Credit Card,Canada,2026-03-04,2026-03-04T16:02:01.197Z
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,Electronics,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:02:01.197Z
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:02:01.197Z
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,Electronics,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:01.197Z


### **Adding total price column using withColumn()**

In [0]:
df = df.withColumn('Total_price',col('unit_price')*col("quantity"))

In [0]:
df.display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate,Total_price
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,Electronics,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:02.032Z,999.99
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,Electronics,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:02:02.032Z,399.98
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,Footwear,1,129.99,Credit Card,Canada,2026-03-04,2026-03-04T16:02:02.032Z,129.99
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,Electronics,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:02:02.032Z,899.99
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,Footwear,2,149.99,UPI,India,2026-03-04,2026-03-04T16:02:02.032Z,299.98
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,Electronics,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:02.032Z,749.99


In [0]:
from pyspark.sql.functions import upper, trim, col

df = df.withColumn("country", upper(trim(col("country")))) \
       .withColumn("customer_name", trim(col("customer_name"))) \
       .withColumn("product_name", trim(col("product_name"))) \
       .withColumn("product_category", upper(trim(col("product_category"))))

In [0]:
df.select("country", "product_category").distinct().display()

country,product_category
USA,ELECTRONICS
CANADA,FOOTWEAR
INDIA,FOOTWEAR


In [0]:
from pyspark.sql.functions import col

df = df.withColumn("unit_price", col("unit_price").cast("double")) \
       .withColumn("Total_price", col("Total_price").cast("double"))

In [0]:
df.display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate,Total_price
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,ELECTRONICS,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:03.692Z,999.99
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,ELECTRONICS,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:02:03.692Z,399.98
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,FOOTWEAR,1,129.99,Credit Card,CANADA,2026-03-04,2026-03-04T16:02:03.692Z,129.99
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,ELECTRONICS,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:02:03.692Z,899.99
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,FOOTWEAR,2,149.99,UPI,INDIA,2026-03-04,2026-03-04T16:02:03.692Z,299.98
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,ELECTRONICS,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:03.692Z,749.99


In [0]:
# Create temporary view
df.createOrReplaceTempView("silver_df_view")

In [0]:
%sql
select * from silver_df_view

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate,Total_price
1001,2026-03-04,1,Alice Johnson,alice@gmail.com,501,iPhone 15,ELECTRONICS,1,999.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:05.151Z,999.99
1002,2026-03-04,2,Bob Smith,bob@yahoo.com,502,AirPods Pro 2,ELECTRONICS,2,199.99,PayPal,USA,2026-03-04,2026-03-04T16:02:05.151Z,399.98
1003,2026-03-04,3,Charlie Brown,charlie@outlook.com,503,Adidas Shoes,FOOTWEAR,1,129.99,Credit Card,CANADA,2026-03-04,2026-03-04T16:02:05.151Z,129.99
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,ELECTRONICS,1,899.99,Debit Card,USA,2026-03-04,2026-03-04T16:02:05.151Z,899.99
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,FOOTWEAR,2,149.99,UPI,INDIA,2026-03-04,2026-03-04T16:02:05.151Z,299.98
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,ELECTRONICS,1,749.99,Credit Card,USA,2026-03-04,2026-03-04T16:02:05.151Z,749.99


### **Adding needed columns into silver table view for scd type 2**

### **MERGE**

### **MERGE USING SQL**

In [0]:
if spark.catalog.tableExists('retailsales.silver.silver_table'):

    # =========================
    # STEP 1 — Expire old rows
    # =========================
    spark.sql("""
        MERGE INTO retailsales.silver.silver_table trg
        USING silver_df_view src
        ON trg.order_id = src.order_id
        AND trg.isCurrent = 'Y'

        WHEN MATCHED AND (
            src.order_id<>trg.order_id or
            src.order_date <>trg.order_date  or
            src.customer_id<>trg.customer_id or
            src.customer_name<>trg.customer_name or
            src.customer_email<>trg.customer_email or
            src.product_id<>trg.product_id or
            src.product_name<>trg.product_name or
            src.product_category <>trg.product_category  or
            src.quantity<>trg.quantity or
            src.unit_price<>trg.unit_price or
            src.payment_type<>trg.payment_type or
            src.country<>trg.country
        )
        THEN UPDATE SET
            trg.endDate = current_timestamp(),
            trg.isCurrent = 'N'
    """)

    # =========================
    # STEP 2 — Insert new active rows
    # =========================
    spark.sql("""
        MERGE INTO retailsales.silver.silver_table trg
        USING silver_df_view src
        ON trg.order_id = src.order_id
        AND trg.isCurrent = 'Y'

        WHEN NOT MATCHED
        THEN INSERT (
            order_id,
            order_date,
            customer_id,
            customer_name,
            customer_email,
            product_id,
            product_name,
            product_category,
            quantity,
            unit_price,
            processDate,
            total_price,  -- add this
            payment_type,
            country,
            last_updated,
            startDate,
            endDate,
            isCurrent
        )
        VALUES (
            src.order_id,
            src.order_date,
            src.customer_id,
            src.customer_name,
            src.customer_email,
            src.product_id,
            src.product_name,
            src.product_category,
            src.quantity,
            src.unit_price,
            src.processDate,
            src.quantity * src.unit_price,  -- calculate here
            src.payment_type,
            src.country,
            src.last_updated,
            current_timestamp(),
            cast('3000-01-01' as timestamp),
            'Y'
        )
    """)

else:
    # Initial load
    spark.sql("""
        CREATE OR REPLACE TABLE retailsales.silver.silver_table AS
        SELECT *,
            current_timestamp() AS startDate,
            cast('3000-01-01' as timestamp) AS endDate,
            'Y' AS isCurrent
        FROM silver_df_view
    """)

In [0]:
%sql
select * from retailsales.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,processDate,Total_price,startDate,endDate,isCurrent
1015,2026-03-01,15,Olivia Martin,olivia@gmail.com,515,Mouse,ELECTRONICS,2,29.99,Credit Card,USA,2026-03-01,2026-03-04T15:57:58.031Z,59.98,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1018,2026-03-01,18,Riya Sharma,riya@gmail.com,518,Smart Watch,ELECTRONICS,1,199.99,UPI,INDIA,2026-03-01,2026-03-04T15:57:58.031Z,199.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1017,2026-03-01,17,Quinn Harris,quinn@gmail.com,517,Tablet,ELECTRONICS,1,399.99,Credit Card,CANADA,2026-03-01,2026-03-04T15:57:58.031Z,399.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1019,2026-03-01,19,Sam Green,sam@gmail.com,519,Jacket,APPAREL,1,89.99,Credit Card,USA,2026-03-01,2026-03-04T15:57:58.031Z,89.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1006,2026-03-01,6,Frank Thomas,frank@yahoo.com,506,HP Laptop,ELECTRONICS,1,749.99,Credit Card,USA,2026-03-01,2026-03-04T15:57:58.031Z,749.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1009,2026-03-01,9,Isabella Clark,isabella@gmail.com,509,iPhone 14,ELECTRONICS,1,999.99,Credit Card,USA,2026-03-01,2026-03-04T15:57:58.031Z,999.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1005,2026-03-01,5,Emma Wilson,unknown,505,Adidas Sneakers,FOOTWEAR,2,149.99,UPI,INDIA,2026-03-01,2026-03-04T15:57:58.031Z,299.98,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1012,2026-03-01,12,Liam Scott,liam@yahoo.com,512,Puma T-Shirt,APPAREL,3,39.99,Cash,USA,2026-03-01,2026-03-04T15:57:58.031Z,119.97,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1004,2026-03-01,4,David Miller,david@gmail.com,504,Samsung S23,ELECTRONICS,1,899.99,Debit Card,USA,2026-03-01,2026-03-04T15:57:58.031Z,899.99,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
1010,2026-03-01,10,Jack Turner,unknown,510,Sony Headphones,ELECTRONICS,1,100.0,Credit Card,AUSTRALIA,2026-03-01,2026-03-04T15:57:58.031Z,100.0,2026-03-04T15:57:58.031Z,3000-01-01T00:00:00.000Z,Y
